In [21]:
!pip install Wikipedia-API pandas requests tqdm beautifulsoup4

Defaulting to user installation because normal site-packages is not writeable


In [22]:
!pip install python-dotenv
import os
from dotenv import load_dotenv
load_dotenv()
MY_EMAIL = os.getenv("WIKI_EMAIL")
if not MY_EMAIL:
    print("Error: MY_EMAIL not found. Make sure your .env file is set up correctly!")
else:
    print("MY_EMAIL loaded securely!")

Defaulting to user installation because normal site-packages is not writeable
MY_EMAIL loaded securely!


In [23]:
import random
import time

import pandas as pd

import requests
import wikipediaapi

import re

from tqdm import tqdm

from urllib.parse import unquote, urljoin, urlparse

from bs4 import BeautifulSoup

In [24]:
base_url = "https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia/all-access/"
headers = {"User-Agent": f"WikiTrendProject1.0 ({MY_EMAIL})"}
def get_yearly_titles(start_year, end_year):
    raw_titles = []

    for year in range(start_year, end_year + 1):
        if year > 2026:
            break

        for month in tqdm(range(1,13),desc=f"Fetching {year}"):
            if year == 2026 and month > 4:
                break
            
            url = f"{base_url}{year}/{month:02d}/all-days"
            try:
                response = requests.get(url, headers=headers, timeout=10)
                if response.status_code == 200:
                    articles = response.json()['items'][0]['articles']
                    for article in articles:
                        title = article['article']
                        if ":" not in title and title != "Main_Page":
                            raw_titles.append({'Title': title,"Views": article['views']})
                    time.sleep(random.uniform(0.5, 1.5))
            except Exception as e:
                    print(f"Error fetching {year}-{month:02d}: {e}")
                    time.sleep(random.uniform(0.5, 1.5))
        if year == 2026:
            break
    if not raw_titles:
        print("No data fetched. Please check your API access and try again.")
        return pd.DataFrame(columns=["Title", "Views"])

    df_raw = pd.DataFrame(raw_titles)
    df_total = df_raw.groupby("Title")["Views"].sum().reset_index()
    df_total = df_total.sort_values(by="Views", ascending=False).reset_index()
    df_top_500 = df_total.head(500)
    print('Most viewed articles:')
    display(df_top_500.head())
    return df_top_500              

In [ ]:
df_final = get_yearly_titles(2024, 2024)

Fetching 2024: 100%|██████████| 12/12 [00:17<00:00,  1.43s/it]

Most viewed articles:


,index,Title,Views
0,1054,Cleopatra,49932153
1,1236,Deaths_in_2024,49860251
2,4876,YouTube,42160485
3,4839,XXXTentacion,36580537
4,3549,Pornhub,31512669


In [26]:
output_dir = "wiki_full_texts"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [27]:
def get_wiki_details(df_titles):
    wiki = wikipediaapi.Wikipedia(
        user_agent=f"WikiTrendProject1.0 ({MY_EMAIL})",
        language='en',
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )

    details = []
    for title in tqdm(df_titles['Title'], desc="Fetching details"):
        try:
            page = wiki.page(title)
            if page.exists():
                full_text = page.text
                safe_title = re.sub(r'[\\/*?:"<>|]', "_", title)
                file_path = os.path.join(output_dir, f"{safe_title}.txt")
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(full_text)
                details.append({
                    'Title': title,
                    'Summary': page.summary[:500],
                    'Text Path': file_path,
                    'URL': page.fullurl
                })
            else:
                details.append({
                    'Title': title,
                    'Summary': 'Page does not exist',
                    'Text Path': '',
                    'URL': ''
                })
            time.sleep(random.uniform(0.5, 1.5))
        except Exception as e:
            print(f"Error fetching details for {title}: {e}")
            time.sleep(random.uniform(0.5, 1.5))
    df_details = pd.DataFrame(details)
    display(df_details.head())
    return df_details

In [28]:
df_final_details = get_wiki_details(df_final)

Fetching details: 100%|██████████| 500/500 [13:49<00:00,  1.66s/it]


,Title,Summary,Text Path,URL
0,Cleopatra,Cleopatra VII Thea Philopator (Koine Greek: Κλ...,wiki_full_texts\Cleopatra.txt,https://en.wikipedia.org/wiki/Cleopatra
1,Deaths_in_2024,This is a list of lists of deaths of significa...,wiki_full_texts\Deaths_in_2024.txt,https://en.wikipedia.org/wiki/Lists_of_deaths_...
2,YouTube,YouTube is an American online video sharing p...,wiki_full_texts\YouTube.txt,https://en.wikipedia.org/wiki/YouTube
3,XXXTentacion,"Jahseh Dwayne Ricardo Onfroy (January 23, 1998...",wiki_full_texts\XXXTentacion.txt,https://en.wikipedia.org/wiki/XXXTentacion
4,Pornhub,Pornhub is a Canadian-owned Internet pornograp...,wiki_full_texts\Pornhub.txt,https://en.wikipedia.org/wiki/Pornhub
